In [ ]:
!which python

In [ ]:
import polars as pl

FILE_PATH = 'binance_top5_data.csv'

In [ ]:
column_names = ['timestamp']
for i in range(1, 6):
    column_names.extend([f"bid_p{i}", f"bid_v{i}"])
for i in range(1, 6):
    column_names.extend([f"ask_p{i}", f"ask_v{i}"])
    
df = pl.scan_csv(
    FILE_PATH,
    has_header=False,
    new_columns=column_names,
    truncate_ragged_lines=True).head(700000)

In [ ]:
df.head(5).collect()

In [ ]:
TICK_SIZE = 0.08
FUTURE_STEPS = 10

q = (
    df
    .with_columns([
        ((pl.col('bid_p1') + pl.col('ask_p1')) / 2.0).alias('mid_price'),

        ((pl.col('bid_v1') - pl.col('ask_v1')) / (pl.col('bid_v1') + pl.col('ask_v1'))).alias('obi')
    ])
    .with_columns([
        pl.col('mid_price').shift(-FUTURE_STEPS).alias('future_mid_price')
    ])
    .with_columns([
        pl.when(pl.col('future_mid_price') > pl.col('mid_price') + TICK_SIZE).then(1)
        .when(pl.col('future_mid_price') < pl.col('mid_price') - TICK_SIZE).then(-1)
        .otherwise(0).alias('target')
    ])
    .drop_nulls()
)

final_df = q.collect()

In [ ]:
final_df.select(['timestamp', 'mid_price', 'obi', 'target']).head(5)

In [ ]:
final_df['target'].value_counts()

In [ ]:
analysis = final_df.group_by('target').agg([
    pl.col('obi').mean().alias('mean_obi'),
    pl.col('obi').std().alias('std_obi'),
    pl.len().alias('count')
]).sort('target')
analysis

In [ ]:
q_top5 = (
    df
    .with_columns([
        pl.sum_horizontal([f"bid_v{i}" for i in range(1, 6)]).alias('total_bid_v'),
        pl.sum_horizontal([f"ask_v{i}" for i in range(1, 6)]).alias('total_ask_v')
    ])
    .with_columns([
        ((pl.col('bid_p1') + pl.col('ask_p1')) / 2.0).alias('mid_price'),
        ((pl.col('total_bid_v') - pl.col('total_ask_v')) 
         / (pl.col('total_bid_v') + pl.col('total_ask_v'))).alias('obi_top5')
    ])
    .with_columns([
        pl.col('mid_price').shift(-FUTURE_STEPS).alias('future_mid_price')
    ])
    .with_columns([
        pl.when(pl.col('future_mid_price') > pl.col('mid_price') + TICK_SIZE).then(1)
        .when(pl.col('future_mid_price') < pl.col('mid_price') - TICK_SIZE).then(-1)
        .otherwise(0).alias('target')
    ])
    .drop_nulls()
)

final_df_top5 = q_top5.collect()

In [ ]:
analysis_top5 = final_df_top5.group_by('target').agg([
    pl.col('obi_top5').mean().alias('mean_obi_top5'),
    pl.col('obi_top5').std().alias('std_obi_top5'),
    pl.len().alias('count')
]).sort('target')
analysis_top5

In [ ]:
# %pip install xgboost scikit-learn pandas

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

In [ ]:
X = final_df.select(['obi']).to_numpy()
y = final_df.select(['target']).to_numpy().ravel()

y = y + 1

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    tree_method='hist',
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Actual -1', 'Actual 0', 'Actual 1'],
                     columns=['Pred -1', 'Pred 0', 'Pred 1'])
cm_df

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Down -1', 'Stay 0', 'Up 1']))

In [ ]:
import numpy as np

In [ ]:
y_probs = model.predict_proba(X_test)

CONFIDENCE_THRESH = 0.60

prob_down = y_probs[:,0]
prob_up = y_probs[:,2]

custom_preds = np.ones(len(y_probs))
custom_preds[prob_down > CONFIDENCE_THRESH] = 0
custom_preds[prob_up > CONFIDENCE_THRESH] = 2

In [ ]:
print(classification_report(y_test, custom_preds, target_names=['Down -1', 'Stay 0', 'Up 1']))

In [ ]:
#%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
#%pip install pyarrow

In [ ]:
plot_df = final_df.tail(2000).to_pandas()

x = np.arange(len(plot_df))
y_mid = plot_df['mid_price']
obi = plot_df['obi']

fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(x, y_mid, label='Mid-Price', color='black', linewidth=1.5, alpha=0.8)

bid_mask = obi > CONFIDENCE_THRESH
ask_mask = obi < -CONFIDENCE_THRESH

ax.scatter(x[bid_mask], y_mid[bid_mask], color='limegreen', marker='^',
           s=120, label='Strong OBI (bid > 0.6)', zorder=5, edgecolor='black')

ax.scatter(x[ask_mask], y_mid[ask_mask], color='crimson', marker='v',
           s=120, label='Strong OBI (ask < -0.6)', zorder=5, edgecolor='black')

ax.set_title('Signal Visualization', fontsize=18, fontweight='bold')
ax.set_xlabel('Tick Index (Time ->)', fontsize=14)
ax.set_ylabel('Mid-Price', fontsize=14)
ax.legend(loc='best', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_df = final_df.tail(1000).to_pandas()

x = np.arange(len(plot_df))
y_mid = plot_df['mid_price']
y_bid = plot_df['bid_p1']
y_ask = plot_df['ask_p1']
obi = plot_df['obi']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [3, 1]})

ax1.fill_between(x, y_bid, y_ask, color='gray', alpha=0.3, label='Spread Channel (Bid-Ask)')
ax1.plot(x, y_mid, label='Mid-Price', color='black', linewidth=2)

buy_mask = obi > 0.6
sell_mask = obi < -0.6
ax1.scatter(x[buy_mask], y_mid[buy_mask], color='limegreen', marker='^', s=150, zorder=5, edgecolor='black', label='Buy Signal (OBI > 0.6)')
ax1.scatter(x[sell_mask], y_mid[sell_mask], color='crimson', marker='v', s=150, zorder=5, edgecolor='black', label='Sell Signal (OBI < -0.6)')

ax1.set_title('Spread Channel and Mid-Price', fontsize=18, fontweight='bold')
ax1.set_ylabel('Price', fontsize=14)
ax1.legend(loc='best', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(x, obi, color='blue', linewidth=1.5, label='OBI L1')
ax2.axhline(0.6, color='limegreen', linestyle='--', linewidth=2)
ax2.axhline(-0.6, color='crimson', linestyle='--', linewidth=2)
ax2.axhline(0, color='black', linestyle='-', linewidth=1)

ax2.fill_between(x, obi, 0.6, where=(obi > 0.6), color='limegreen', alpha=0.5)
ax2.fill_between(x, obi, -0.6, where=(obi < -0.6), color='crimson', alpha=0.5)

ax2.set_title('Order Book Imbalance OBI Pulse', fontsize=14, fontweight='bold')
ax2.set_xlabel('Tick Index (Time ->)', fontsize=14)
ax2.set_ylabel('OBI Value', fontsize=14)
ax2.set_ylim(-1.1, 1.1)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
q_v2 = (
    df
    .with_columns([
        pl.sum_horizontal([f"bid_v{i}" for i in range(1, 6)]).alias('total_bid_v'),
        pl.sum_horizontal([f"ask_v{i}" for i in range(1, 6)]).alias('total_ask_v')
    ])
    .with_columns([
        ((pl.col('bid_p1') + pl.col('ask_p1')) / 2.0).alias('mid_price'),
        ((pl.col('bid_v1') - pl.col('ask_v1')) / (pl.col('bid_v1') + pl.col('ask_v1'))).alias('obi'),
        
        ((pl.col('total_bid_v') - pl.col('total_ask_v')) / 
         (pl.col('total_bid_v') + pl.col('total_ask_v'))).alias('obi_top5'),
         
        (pl.col('ask_p1') - pl.col('bid_p1')).alias('spread')
    ])
    .with_columns([
        pl.col('mid_price').shift(-FUTURE_STEPS).alias('future_mid_price')
    ])
    .with_columns([
        pl.when(pl.col('future_mid_price') > pl.col('mid_price') + TICK_SIZE).then(1)
        .when(pl.col('future_mid_price') < pl.col('mid_price') - TICK_SIZE).then(-1)
        .otherwise(0).alias('target')
    ])
    .drop_nulls()
)

In [ ]:
df_v2 = q_v2.collect()
df_v2.shape

In [ ]:
X_v2 = df_v2.select(['obi', 'obi_top5', 'spread']).to_numpy()
y_v2 = df_v2.select(['target']).to_numpy().ravel()

y_v2 = y_v2 + 1

split = int(len(X_v2) * 0.8)
X_v2_train, X_v2_test = X_v2[:split], X_v2[split:]
y_v2_train, y_v2_test = y_v2[:split], y_v2[split:]

In [ ]:
from xgboost import XGBClassifier

model_v2 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

model_v2.fit(X_v2_train, y_v2_train)

In [ ]:
y_v2_pred = model_v2.predict(X_v2_test)

cm_v2 = confusion_matrix(y_v2_test, y_v2_pred)
cm_v2_df = pd.DataFrame(cm_v2, index=['Actual -1', 'Actual 0', 'Actual 1'],
                     columns=['Pred -1', 'Pred 0', 'Pred 1'])
cm_v2_df

In [ ]:
y_v2_probs = model_v2.predict_proba(X_v2_test)

CONFIDENCE_THRESH_v2 = 0.60

prob_v2_down = y_v2_probs[:,0]
prob_v2_up = y_v2_probs[:,2]

custom_v2_preds = np.ones(len(y_v2_probs))
custom_v2_preds[prob_v2_down > CONFIDENCE_THRESH_v2] = 0
custom_v2_preds[prob_v2_up > CONFIDENCE_THRESH_v2] = 2

In [ ]:
print(classification_report(y_v2_test, custom_v2_preds, target_names=['Down -1', 'Stay 0', 'Up 1']))

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

model_v2.get_booster().feature_names = ['obi_l1', 'obi_top5', 'spread']

fig, ax = plt.subplots(figsize=(10, 5))
plot_importance(model_v2, ax=ax, importance_type='weight', 
                title='Feature Importance',
                color='royalblue')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
plot_importance(model_v2, ax=ax, importance_type='gain', 
                title='Feature Importance',
                color='crimson')
plt.show()

In [ ]:
corr_v2_matrix = df_v2.select(['obi', 'obi_top5', 'spread']).to_pandas().corr()
corr_v2_matrix

In [ ]:
q_v3 = (
    df
    .with_columns([
        ((pl.col('bid_p1') + pl.col('ask_p1')) / 2.0).alias('mid_price'),
        ((pl.col('bid_v1') - pl.col('ask_v1')) / (pl.col('bid_v1') + pl.col('ask_v1'))).alias('obi')
    ])
    .with_columns([
        (pl.col('mid_price') - pl.col('mid_price').shift(10)).alias('momentum_10'),
        (pl.col('mid_price') - pl.col('mid_price').shift(50)).alias('momentum_50')
    ])
    .with_columns([
        pl.col('mid_price').shift(-FUTURE_STEPS).alias('future_mid_price')
    ])
    .with_columns([
        pl.when(pl.col('future_mid_price') > pl.col('mid_price') + TICK_SIZE).then(1)
        .when(pl.col('future_mid_price') < pl.col('mid_price') - TICK_SIZE).then(-1)
        .otherwise(0).alias('target')
    ])
    .drop_nulls()
)

df_v3 = q_v3.collect()
df_v3.shape

In [ ]:
X_v3 = df_v3.select(['obi', 'momentum_10', 'momentum_50']).to_numpy()
y_v3 = df_v3.select(['target']).to_numpy().ravel()

y_v3 = y_v3 + 1

split = int(len(X_v3) * 0.8)
X_v3_train, X_v3_test = X_v3[:split], X_v3[split:]
y_v3_train, y_v3_test = y_v3[:split], y_v3[split:]

In [ ]:
from xgboost import XGBClassifier

model_v3 = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

model_v3.fit(X_v3_train, y_v3_train)

In [ ]:
y_v3_pred = model_v3.predict(X_v3_test)

cm_v3 = confusion_matrix(y_v3_test, y_v3_pred)
cm_v3_df = pd.DataFrame(cm_v3, index=['Actual -1', 'Actual 0', 'Actual 1'],
                     columns=['Pred -1', 'Pred 0', 'Pred 1'])
cm_v3_df

In [ ]:
y_v3_probs = model_v3.predict_proba(X_v3_test)

CONFIDENCE_THRESH_v3 = 0.60

prob_v3_down = y_v3_probs[:,0]
prob_v3_up = y_v3_probs[:,2]

custom_v3_preds = np.ones(len(y_v3_probs))
custom_v3_preds[prob_v3_down > CONFIDENCE_THRESH_v3] = 0
custom_v3_preds[prob_v3_up > CONFIDENCE_THRESH_v3] = 2

In [ ]:
print(classification_report(y_v3_test, custom_v3_preds, target_names=['Down -1', 'Stay 0', 'Up 1']))

In [ ]:
corr_v3_matrix = df_v3.select(['obi', 'momentum_10', 'momentum_50']).to_pandas().corr()
corr_v3_matrix

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

model_v3.get_booster().feature_names = ['obi', 'momentum_10', 'momentum_50']

fig, ax = plt.subplots(figsize=(10, 5))
plot_importance(model_v3, ax=ax, importance_type='gain', 
                title='Feature Importance',
                color='crimson')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

signals = custom_v3_preds - 1 

split_idx = int(len(df_v3) * 0.8)
test_df = df_v3[split_idx:].to_pandas()

price_diff = test_df['future_mid_price'] - test_df['mid_price']

FEE_PER_TRADE = 0.005 

gross_pnl = price_diff * signals

is_trade = (signals != 0).astype(int)
net_pnl = gross_pnl - (is_trade * FEE_PER_TRADE)

equity_curve = net_pnl.cumsum()

plt.figure(figsize=(16, 8))
plt.plot(equity_curve.values, color='limegreen', linewidth=2, label='Equity Curve (Net PnL)')
plt.title('Sniper v3 ahhhh model', fontsize=18, fontweight='bold')
plt.xlabel('Tick Index (Oś czasu)', fontsize=14)
plt.ylabel('Net PnL', fontsize=14)

plt.fill_between(range(len(equity_curve)), equity_curve.values, 0, where=(equity_curve.values > 0), color='limegreen', alpha=0.1)
plt.fill_between(range(len(equity_curve)), equity_curve.values, 0, where=(equity_curve.values < 0), color='crimson', alpha=0.1)

plt.axhline(0, color='black', linewidth=1.5, linestyle='--')
plt.legend(loc='upper left', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

total_trades = is_trade.sum()
winning_trades = np.sum(net_pnl > 0)
losing_trades = np.sum(net_pnl < 0)
win_rate = winning_trades / total_trades if total_trades > 0 else 0
final_pnl = equity_curve.iloc[-1] if len(equity_curve) > 0 else 0

print(f"Number of Transactions: {total_trades}")
print(f"Profitable trades:     {winning_trades}")
print(f"Losing Trades:       {losing_trades}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Net PnL: ${final_pnl:.2f}")

In [ ]:
import os

model_filename = "sniper_v3.json"

model_v3.save_model(model_filename)

file_size_mb = os.path.getsize(model_filename) / (1024 * 1024)

print(f"File: {model_filename}")
print(f"Size on the disk: {file_size_mb:.2f} MB")